# LlamaIndex: documentos, indice vectorial y RAG

LlamaIndex esta pensado para armar sistemas de recuperacion sobre documentos.

En este notebook usamos el mismo dataset y la misma pregunta.

| Concepto | En LlamaIndex |
|---|---|
| Documento | `Document` |
| Chunking | `SentenceSplitter` |
| Embeddings | `OpenAIEmbedding` |
| Indice vectorial | `VectorStoreIndex` |
| Busqueda Top-K | `as_retriever(similarity_top_k=3)` |
| RAG | `as_query_engine()` |

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from dotenv import load_dotenv
from llama_index.core import Document, Settings, VectorStoreIndex
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI

BASE = Path.cwd()
if not (BASE / 'data' / 'documentos.py').exists(): BASE = BASE.parent
if not (BASE / 'data' / 'documentos.py').exists(): BASE = Path('ai_engineer/ejercicios_embeddings_simple').resolve()
load_dotenv(BASE.parent / '.env'); sys.path.append(str(BASE / 'data'))

from documentos import DOCUMENTOS, PREGUNTA

## 1. Configuracion global

LlamaIndex usa `Settings` para saber que modelos usar.

| Setting | Para que sirve |
|---|---|
| `embed_model` | Modelo que convierte texto en embeddings |
| `llm` | Modelo que redacta la respuesta final |
| `node_parser` | Estrategia para dividir documentos en chunks |

En LlamaIndex muchas veces a los chunks se les dice **nodes**.

In [ ]:
# OpenAIEmbedding docs: https://docs.llamaindex.ai/en/stable/examples/embeddings/OpenAI/
Settings.embed_model = OpenAIEmbedding(model='text-embedding-3-small')
Settings.llm = OpenAI(model='gpt-4o-mini', temperature=0)
# SentenceSplitter docs: https://docs.llamaindex.ai/en/stable/module_guides/loading/node_parsers/modules/#sentencesplitter
Settings.node_parser = SentenceSplitter(chunk_size=120, chunk_overlap=20)

print('Embedding model:', Settings.embed_model.model_name)
print('LLM:', Settings.llm.model)
print('Chunk size:', Settings.node_parser.chunk_size)
print('Chunk overlap:', Settings.node_parser.chunk_overlap)

## 2. Documentos

Convertimos cada diccionario del dataset en un `Document` de LlamaIndex.

La metadata `source` nos permite saber de que archivo vino cada resultado.

In [ ]:
# Document docs: https://docs.llamaindex.ai/en/stable/module_guides/loading/documents_and_nodes/
docs = [
    Document(text=doc['text'], metadata={'source': doc['source']})
    for doc in DOCUMENTOS
]

for doc in docs:
    print(f"--- {doc.metadata['source']} ---")
    print(doc.text.strip(), '\n')

## 3. Indice vectorial

`VectorStoreIndex.from_documents(docs)` hace varias cosas por nosotros:

1. Divide documentos en nodes/chunks.
2. Crea embeddings de esos chunks.
3. Los guarda en un indice vectorial.

Es una version mas automatica del pipeline manual.

In [ ]:
# from_documents hace todo junto: crea nodes, embeddings, e indice vectorial.
# VectorStoreIndex docs: https://docs.llamaindex.ai/en/stable/module_guides/indexing/vector_store_index/
index = VectorStoreIndex.from_documents(docs)

nodes = list(index.docstore.docs.values())
for i, node in enumerate(nodes, start=1):
    print(f"{i}. ({node.metadata.get('source', '')}) {node.get_content()}")

### Grafico: longitud de nodes

Aca vemos cuantos terminos aproximados tiene cada node creado por LlamaIndex.

In [ ]:
longitudes = [len(node.get_content().split()) for node in nodes]

plt.figure(figsize=(8, 3))
plt.bar(range(1, len(nodes) + 1), longitudes)
plt.title('Palabras por node en LlamaIndex')
plt.xlabel('Node')
plt.ylabel('Palabras')
plt.show()

## 4. Busqueda Top-K

El retriever consulta el indice vectorial.

`similarity_top_k=3` significa que queremos los 3 nodes mas parecidos a la pregunta.

In [ ]:
# Le pedimos al indice los 3 nodes mas parecidos a la pregunta.
# Retriever docs: https://docs.llamaindex.ai/en/stable/module_guides/querying/retriever/
top_3 = index.as_retriever(similarity_top_k=3).retrieve(PREGUNTA)

print('Pregunta:', PREGUNTA)
for item in top_3:
    print(f"[{item.score:.4f}] ({item.node.metadata['source']}) {item.node.get_content()}")

### Grafico: scores Top-K

Los scores muestran que tan fuerte fue la relacion entre la pregunta y cada node recuperado.

In [ ]:
plt.figure(figsize=(6, 3))
plt.bar([item.node.metadata['source'] for item in top_3], [item.score for item in top_3])
plt.title('Scores Top-3 en LlamaIndex')
plt.ylabel('Score')
plt.xticks(rotation=20)
plt.show()

## 5. RAG con query engine

`as_query_engine()` une dos etapas:

1. Retrieval: busca contexto relevante.
2. Generation: genera una respuesta con el LLM.

Por eso esta celda es mas corta que en el notebook manual.

In [ ]:
# El query engine hace retrieval + generation en un solo paso.
# Docs: https://docs.llamaindex.ai/en/stable/module_guides/deploying/query_engine/
query_engine = index.as_query_engine(similarity_top_k=3)
respuesta = query_engine.query(PREGUNTA)

print('Respuesta RAG:', respuesta)

## Cierre

LlamaIndex automatiza gran parte del pipeline de RAG, pero los conceptos son los mismos:

- documentos;
- chunks/nodes;
- embeddings;
- indice vectorial;
- busqueda Top-K;
- respuesta con contexto.